# 01. Análise exploratória (EDA)

**Objetivo:** diagnosticar as quatro bases brutas (estrutura, nulos, duplicatas, outliers, distribuições e comportamento no tempo) sem usar inadimplência. Aqui só se diagnostica; o tratamento fica para outro notebook.

In [ ]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.functions import auditar_categoricas
import pandas as pd
import numpy as np

In [ ]:
cadastral = pd.read_parquet("../dados/base_cadastral.parquet")
submissao = pd.read_parquet("../dados/base_submissao.parquet")
emprestimos = pd.read_parquet("../dados/historico_emprestimos.parquet")
parcelas = pd.read_parquet("../dados/historico_parcelas.parquet")

for nome, df in [("cadastral", cadastral), ("submissao", submissao),
                 ("emprestimos", emprestimos), ("parcelas", parcelas)]:
    print(f"{nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

## Base Cadastral

In [ ]:
cadastral.info()

In [ ]:
cadastral.head()

In [ ]:
# Aplicando função criada para fazer uma auditoria profunda em dados categóricos (excluindo colunas com tipos incorretos).
resumo, freq, variantes = auditar_categoricas(
    cadastral, excluir=["data_nascimento", "renda_anual"]
)
resumo

In [ ]:
freq

In [ ]:
variantes

In [ ]:
cadastral.describe()

In [ ]:
# Avaliando estrutura de colunas que deveriam ter tipos diferents:
print(cadastral["data_nascimento"].str.replace(r"\d", "9", regex=True).value_counts(dropna=False).head(10))
print(cadastral["renda_anual"].str.replace(r"\d", "", regex=True).value_counts(dropna=False).head(10))
cadastral[["data_nascimento", "renda_anual"]].sample(8, random_state=42)

### Decisões para `cadastral` (variáveis categóricas)

- `sexo`, `tipo_renda`, `possui_carro`, `possui_imovel`: padronizar caixa, acentos e espaços para um rótulo canônico (Feminino, Masculino, Sim, Não, Assalariado, Autônomo).
- Placeholder "Não informado" em `sexo`, `possui_carro` e `possui_imovel` (~0,1% cada): converter para nulo e decidir a imputação no pipeline, ajustada só no treino.
- `nivel_educacao`, `tipo_moradia`, `ocupacao` (categorias): sem variantes de digitação.
- Pendentes: tratamento do placeholder em `tipo_renda`, `tipo_organizacao` e `estado_civil`; significado dos nulos de `ocupacao`; categorias raras.
- Há colunas que deverão ser excluídas antes do merge com as demais, como `sexo` (dados sensíveis) e `qtd_filhos` (redundante)

### Decisões de tipos para `cadastral`

- `data_nascimento`: dois formatos, `AAAA-MM-DD` (50.562) e `DD/MM/AAAA` (38, ordem confirmada porque o 1º campo passa de 12). Converter com `["%Y-%m-%d", "%d/%m/%Y"]`.
- `renda_anual`: texto com ponto de milhar e vírgula decimal, 101 valores com prefixo `R$` e 101 com "não informado" (viram nulo). Converter com `decimal=","` e `milhar="."`.
- `renda_anual`: 67 valores negativos, a tratar como inconsistência depois da conversão.
- `qtd_membros_familia`: float, converter para `Int64` se não houver valores fracionários.

## Base Cadastral